#ZERO-DAY CYBERATTACK DETECTION

A Hybrid Supervised and Unsupervised Machine Learning




In [ ]:
!pip install -q xgboost imbalanced-learn joblib

# Data processing
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (
    accuracy_score,

    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

from xgboost import XGBClassifier

# Utilities
import os
import glob
import warnings
import joblib
import time

warnings.filterwarnings("ignore")

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Display configuration
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully!")
print("Project environment is ready.")

: 

In [ ]:
# STEP 1: SET UP LOCAL DATASET PATHS

from pathlib import Path
import os
import glob

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "datasets").is_dir():
    PROJECT_DIR = PROJECT_DIR.parent

DATASET_DIR = PROJECT_DIR / "datasets"
if not DATASET_DIR.is_dir():
    raise FileNotFoundError(f"Dataset folder not found: {DATASET_DIR.resolve()}")

print(f"Using local dataset folder: {DATASET_DIR.resolve()}")

In [ ]:
# STEP 2: LOCATE CIC-IDS2017 DATASET

import os
import glob

# Find CSV files in the local datasets folder
csv_files = sorted(glob.glob(os.path.join(DATASET_DIR, "*.csv")))

# Display discovered CSV files
print("=" * 65)
print("CIC-IDS2017 DATASET FILE DISCOVERY")
print("=" * 65)

if len(csv_files) == 0:
    print(f"No CSV files found in {DATASET_DIR.resolve()}.")
else:
    for i, file in enumerate(csv_files, 1):
        size_mb = os.path.getsize(file) / (1024 * 1024)

        print(f"{i}. {os.path.basename(file)}")
        print(f"   Size: {size_mb:.2f} MB")
        print(f"   Location: {file}")
        print()

    print("=" * 65)
    print(f"Total CSV files discovered: {len(csv_files)}")

In [ ]:
# STEP 3: VERIFY DATASET FILES

print("DATASET VERIFICATION")
print("-" * 50)

for file in csv_files:
    print(os.path.basename(file), "->",
          "EXISTS" if os.path.isfile(file) else "MISSING")

print("-" * 50)

print("Total files:", len(csv_files))
print("Total dataset size:",
      round(sum(os.path.getsize(f) for f in csv_files)
            / (1024 ** 2), 2), "MB")

In [ ]:
# STEP 4: DATASET STRUCTURE INSPECTION

import os
import glob
import pandas as pd
import numpy as np

# Use the local datasets folder configured in STEP 1.
csv_files = sorted(
    glob.glob(os.path.join(DATASET_DIR, "*.csv"))
)

print("=" * 75)
print("CIC-IDS2017 DATASET STRUCTURE ANALYSIS")
print("=" * 75)

assert len(csv_files) == 8, (
    f"Expected 8 CSV files, but found {len(csv_files)}. "
    "Check DATASET_DIR."
)

dataset_info = []

for file in csv_files:

    # Read only 5 rows to inspect the structure
    sample = pd.read_csv(
        file,
        nrows=5,
        encoding="utf-8",
        low_memory=False
    )

    # Clean column names for inspection
    sample.columns = sample.columns.str.strip()

    file_size = os.path.getsize(file) / (1024 ** 2)

    label_column = next(
        (c for c in sample.columns if c.lower() == "label"),
        None
    )

    print("\nFile:", os.path.basename(file))
    print("Size:", round(file_size, 2), "MB")
    print("Number of features:", len(sample.columns))

    if label_column:
        print("Label examples:",
              sample[label_column].unique())
    else:
        print("WARNING: Label column not found!")

    dataset_info.append({
        "File": os.path.basename(file),
        "Size_MB": round(file_size, 2),
        "Features": len(sample.columns),
        "Label_Column": label_column
    })

print("\n" + "=" * 75)
print("DATASET INSPECTION SUMMARY")
print("=" * 75)

display(pd.DataFrame(dataset_info))

In [ ]:
# STEP 5: COMPLETE DATASET LABEL DISTRIBUTION ANALYSIS

import pandas as pd
import numpy as np
import os
import glob
from collections import Counter

print("=" * 75)
print("CIC-IDS2017 COMPLETE LABEL DISTRIBUTION ANALYSIS")
print("=" * 75)

label_counts = Counter()
total_records = 0
file_statistics = []

CHUNK_SIZE = 100_000

for file in csv_files:

    filename = os.path.basename(file)

    print(f"\nProcessing: {filename}")

    file_count = Counter()
    file_rows = 0

    try:
        for chunk in pd.read_csv(
            file,
            chunksize=CHUNK_SIZE,
            encoding="utf-8",
            low_memory=False
        ):

            # Normalize column names
            chunk.columns = (
                chunk.columns.astype(str)
                .str.replace('\ufeff', '', regex=False)
                .str.strip()
            )

            # Identify label column
            label_col = next(
                (c for c in chunk.columns
                 if c.lower() == "label"),
                None
            )

            if label_col is None:
                raise ValueError(
                    f"Label column missing in {filename}"
                )

            # Clean labels
            labels = (
                chunk[label_col]
                .astype(str)
                .str.strip()
                .str.upper()
            )

            counts = Counter(labels)

            file_count.update(counts)
            label_counts.update(counts)

            file_rows += len(chunk)

        total_records += file_rows

        file_statistics.append({
            "File": filename,
            "Total Records": file_rows,
            "Unique Labels": len(file_count)
        })

        print("Records:", f"{file_rows:,}")
        print("Attack categories:", len(file_count))

    except Exception as e:
        print("ERROR:", filename, e)
        raise

print("\n" + "=" * 75)
print("FINAL DATASET SUMMARY")
print("=" * 75)

print("Total records:", f"{total_records:,}")
print("Total attack categories:", len(label_counts))

# Create label distribution table
label_df = pd.DataFrame(
    label_counts.items(),
    columns=["Attack Category", "Records"]
)

label_df = label_df.sort_values(
    by="Records",
    ascending=False
).reset_index(drop=True)

label_df["Percentage"] = (
    label_df["Records"] / total_records * 100
).round(4)

display(label_df)

print("\nFile-wise statistics:")
display(pd.DataFrame(file_statistics))

In [ ]:
# STEP 6: ATTACK DISTRIBUTION VISUALIZATION

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(14, 7))

sns.barplot(
    data=label_df,
    x="Attack Category",
    y="Records"
)

plt.title(
    "CIC-IDS2017 Network Traffic Distribution",
    fontsize=16
)

plt.xlabel("Attack Category")
plt.ylabel("Number of Network Records")

plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()


# Normal vs malicious traffic

benign_count = label_counts.get("BENIGN", 0)

attack_count = (
    total_records - benign_count
)

plt.figure(figsize=(7, 7))

plt.pie(
    [benign_count, attack_count],
    labels=["Benign Traffic", "Malicious Traffic"],
    autopct="%1.2f%%",
    startangle=90
)

plt.title("Benign vs Malicious Network Traffic")
plt.show()

print("Benign records:", f"{benign_count:,}")
print("Malicious records:", f"{attack_count:,}")

In [ ]:
# STEP 7: MEMORY-EFFICIENT DATASET LOADING
# ZERO-DAY CYBERATTACK DETECTION

import pandas as pd
import numpy as np
import os
import glob
import gc
import re

from collections import defaultdict

# Configuration
RANDOM_STATE = 42
CHUNK_SIZE = 100_000

# Sampling limits
SAMPLE_LIMITS = {
    "BENIGN": 250_000,
    "PORTSCAN": 120_000
}

# Maximum samples for each known attack category
DEFAULT_ATTACK_LIMIT = 25_000

# Columns that should not be used as predictive features
DROP_COLUMNS = [
    "FLOW ID",
    "SOURCE IP",
    "DESTINATION IP",
    "TIMESTAMP"
]

# Storage
sampled_data = defaultdict(list)

print("=" * 75)
print("LOADING CIC-IDS2017 DATASET")
print("=" * 75)

def normalize_label(value):

    value = str(value).strip().upper()

    # Normalize encoding artifacts and whitespace
    value = value.replace("\ufffd", " ")
    value = value.replace("\u2013", " ")
    value = value.replace("\u2014", " ")

    value = re.sub(r"\s+", " ", value)

    return value


for file in csv_files:

    print("\nReading:", os.path.basename(file))

    for chunk in pd.read_csv(
        file,
        chunksize=CHUNK_SIZE,
        encoding="latin1",
        low_memory=False
    ):

        # Clean column names
        chunk.columns = (
            chunk.columns.astype(str)
            .str.replace("\ufeff", "", regex=False)
            .str.strip()
            .str.upper()
        )

        if "LABEL" not in chunk.columns:
            raise ValueError(
                f"Label column missing in {file}"
            )

        # Normalize labels
        chunk["LABEL"] = (
            chunk["LABEL"].map(normalize_label)
        )

        # Process each category independently
        for label, group in chunk.groupby("LABEL"):

            limit = SAMPLE_LIMITS.get(
                label,
                DEFAULT_ATTACK_LIMIT
            )

            # Sample within each chunk to control memory
            if len(group) > 5000:
                group = group.sample(
                    n=5000,
                    random_state=RANDOM_STATE
                )

            sampled_data[label].append(group)

    print("Completed:", os.path.basename(file))


print("\nCombining sampled records...")

final_samples = []

for label, frames in sampled_data.items():

    combined = pd.concat(
        frames,
        ignore_index=True
    )

    limit = SAMPLE_LIMITS.get(
        label,
        DEFAULT_ATTACK_LIMIT
    )

    if len(combined) > limit:
        combined = combined.sample(
            n=limit,
            random_state=RANDOM_STATE
        )

    final_samples.append(combined)

    print(
        f"{label}: {len(combined):,} records"
    )


# Combine all categories
df = pd.concat(
    final_samples,
    ignore_index=True
)

del sampled_data, final_samples
gc.collect()

# Shuffle dataset
df = df.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

print("\n" + "=" * 75)
print("DATASET LOADING COMPLETED")
print("=" * 75)

print("Total sampled records:", f"{len(df):,}")
print("Total columns:", len(df.columns))

print("\nAttack distribution:")

display(
    df["LABEL"]
    .value_counts()
    .to_frame("Records")
)

In [ ]:
# STEP 8: DATA CLEANING AND FEATURE PREPARATION

import pandas as pd
import numpy as np
import gc

print("=" * 75)
print("DATA CLEANING AND FEATURE PREPARATION")
print("=" * 75)

# 1. Create a working copy
df_clean = df.copy()

# 2. Clean column names
df_clean.columns = (
    df_clean.columns.astype(str)
    .str.replace('\ufeff', '', regex=False)
    .str.strip()
    .str.upper()
)

# Remove duplicate column names
df_clean = df_clean.loc[
    :, ~df_clean.columns.duplicated()
]

print("\nOriginal sampled shape:", df.shape)

# 3. Normalize attack labels
def clean_attack_label(value):

    value = str(value).strip().upper()

    value = value.replace("Ï¿½", " ")
    value = value.replace("�", " ")
    value = value.replace("\ufffd", " ")

    value = " ".join(value.split())

    # Standardize known CIC-IDS2017 labels
    if "BRUTE FORCE" in value:
        return "WEB ATTACK - BRUTE FORCE"

    elif "SQL INJECTION" in value:
        return "WEB ATTACK - SQL INJECTION"

    elif value.endswith("XSS"):
        return "WEB ATTACK - XSS"

    elif "INFILTRATION" in value:
        return "INFILTRATION"

    return value


df_clean["LABEL"] = (
    df_clean["LABEL"].apply(clean_attack_label)
)

print("\nNormalized attack categories:")

print(df_clean["LABEL"].value_counts())

# 4. Remove non-predictive identifier columns
identifier_columns = [
    "FLOW ID",
    "SOURCE IP",
    "DESTINATION IP",
    "TIMESTAMP"
]

existing_identifiers = [
    col for col in identifier_columns
    if col in df_clean.columns
]

df_clean.drop(
    columns=existing_identifiers,
    inplace=True
)

print("\nRemoved identifier columns:")
print(existing_identifiers)

# 5. Separate target variable
y = df_clean["LABEL"].copy()

X = df_clean.drop(
    columns=["LABEL"]
).copy()

# 6. Convert features to numeric
for col in X.columns:

    X[col] = pd.to_numeric(
        X[col],
        errors="coerce"
    )


# 7. Replace infinite valueS
X.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

# 8. Remove completely empty columns
empty_columns = X.columns[
    X.isna().all()
].tolist()

X.drop(
    columns=empty_columns,
    inplace=True
)

# 9. Remove constant features
constant_columns = [
    col for col in X.columns
    if X[col].nunique(dropna=True) <= 1
]

X.drop(
    columns=constant_columns,
    inplace=True
)


# 10. Remove features with excessive missing values
missing_percent = X.isna().mean() * 100

high_missing_columns = missing_percent[
    missing_percent > 50
].index.tolist()

X.drop(
    columns=high_missing_columns,
    inplace=True
)


# 11. Final dataset
print("\n" + "=" * 75)
print("DATA CLEANING SUMMARY")
print("=" * 75)

print("Original records:", len(df_clean))
print("Remaining features:", X.shape[1])

print("Empty columns removed:", len(empty_columns))
print("Constant columns removed:", len(constant_columns))
print("High-missing columns removed:",
      len(high_missing_columns))

print("Remaining missing values:", X.isna().sum().sum())

print("\nFinal feature matrix shape:", X.shape)
print("Target variable shape:", y.shape)

print("\nFinal attack distribution:")

display(
    y.value_counts().to_frame("Records")
)

print("\nData cleaning completed successfully!")

gc.collect()

In [ ]:
# STEP 9: FEATURE QUALITY AND IDENTIFIER INSPECTION

print("=" * 75)
print("FEATURE INSPECTION")
print("=" * 75)

# Display all available features
print("\nAVAILABLE FEATURES:")

for i, col in enumerate(X.columns, 1):
    print(f"{i:02d}. {col}")

# Check potentially identifying columns
print("\nIDENTIFIER COLUMN CHECK:")

identifier_keywords = [
    "FLOW ID",
    "SOURCE IP",
    "DESTINATION IP",
    "TIMESTAMP",
    "IP ADDRESS"
]

possible_identifiers = [
    col for col in X.columns
    if any(keyword in col.upper()
           for keyword in identifier_keywords)
]

print("Potential identifier columns:",
      possible_identifiers)

# Missing-value analysis
missing_summary = (
    X.isnull()
    .sum()
    .sort_values(ascending=False)
)

print("\nMISSING VALUE SUMMARY:")

display(
    missing_summary[missing_summary > 0]
    .to_frame("Missing Values")
)

# Feature statistics
print("\nFEATURE STATISTICS:")

display(X.describe().T)

print("\nTotal usable features:", X.shape[1])
print("Total records:", len(X))

In [ ]:
# STEP 10: CORRECTED ZERO-DAY EXPERIMENTAL DATASET
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
ZERO_DAY_ATTACK = "PORTSCAN"

print("=" * 70)
print("CORRECTED ZERO-DAY DATASET PREPARATION")
print("=" * 70)

# 1. Start from original cleaned features
X_exp = X.copy().reset_index(drop=True)
labels = y.copy().reset_index(drop=True)

# 2. Remove exact duplicate records, considering labels
combined = X_exp.copy()
combined["ORIGINAL_LABEL"] = labels

before = len(combined)

combined.drop_duplicates(
    keep="first",
    inplace=True
)

combined.reset_index(drop=True, inplace=True)

print("\nDuplicate records removed:", before - len(combined))

labels = combined["ORIGINAL_LABEL"].copy()

X_exp = combined.drop(
    columns=["ORIGINAL_LABEL"]
).copy()

# 3. Convert physically invalid negative measurements
# and CICFlowMeter -1 sentinel values into missing values.

negative_summary = (X_exp < 0).sum()

print("\nNegative values before correction:")
display(
    negative_summary[negative_summary > 0]
    .to_frame("Count")
)

# These fields can contain -1 sentinel values.
# We retain the columns and impute missing values later.
# Other negative measurements are also treated as invalid.

X_exp = X_exp.mask(X_exp < 0, np.nan)

print("\nNegative values remaining:", int((X_exp < 0).sum().sum()))

# 4. Separate unseen PortScan traffic

zero_day_mask = labels == ZERO_DAY_ATTACK

X_zero_day = X_exp.loc[zero_day_mask].copy()
y_zero_day = labels.loc[zero_day_mask].copy()

X_known = X_exp.loc[~zero_day_mask].copy()
y_known = labels.loc[~zero_day_mask].copy()

# 5. Binary target labels
# 0 = BENIGN
# 1 = ATTACK

y_known_binary = (
    y_known != "BENIGN"
).astype(int)

y_zero_day_binary = np.ones(
    len(X_zero_day),
    dtype=int
)

# 6. Stratified split of known traffic

(
    X_train,
    X_test_known,
    y_train,
    y_test_known
) = train_test_split(
    X_known,
    y_known_binary,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_known_binary
)

# Preserve original attack labels

known_labels_train = y_known.loc[
    X_train.index
].copy()

known_labels_test = y_known.loc[
    X_test_known.index
].copy()

# 7. Create final testing dataset

X_test = pd.concat(
    [X_test_known, X_zero_day],
    axis=0
)

y_test = np.concatenate([
    y_test_known.to_numpy(),
    y_zero_day_binary
])

# 8. Isolation Forest receives benign training traffic only

X_if_train = X_train.loc[
    y_train == 0
].copy()

# 9. Verify the experiment

assert ZERO_DAY_ATTACK not in set(
    known_labels_train
)

assert ZERO_DAY_ATTACK not in set(
    known_labels_test
)

assert len(X_zero_day) > 0

assert (y_train.loc[X_if_train.index] == 0).all()

print("\n" + "=" * 70)
print("CORRECTED EXPERIMENT SUMMARY")
print("=" * 70)

print("Training records:", len(X_train))
print("Known testing records:", len(X_test_known))
print("Unseen PortScan records:", len(X_zero_day))
print("Final testing records:", len(X_test))

print("Features:", X_train.shape[1])
print("Isolation Forest training records:", len(X_if_train))

print("\nTraining distribution:")
display(
    y_train.value_counts().rename({
        0: "BENIGN",
        1: "KNOWN ATTACK"
    }).to_frame("Records")
)

print("\nTesting distribution:")
display(
    pd.Series(y_test).value_counts().rename({
        0: "BENIGN",
        1: "ATTACK"
    }).to_frame("Records")
)

print("\nSUCCESS: Zero-day attack excluded from supervised training.")
print("SUCCESS: Isolation Forest uses benign training data only.")
print("SUCCESS: Experimental datasets regenerated.")

In [ ]:
import numpy as np
import pandas as pd

datasets = {
    "Training": X_train,
    "Known Test": X_test_known,
    "PortScan Test": X_zero_day
}

for name, data in datasets.items():
    numeric = data.select_dtypes(include=[np.number])

    inf_count = np.isinf(numeric.to_numpy(dtype=float)).sum()

    print(f"\n{name}")
    print("Shape:", data.shape)
    print("Infinite values:", inf_count)

    if inf_count:
        counts = pd.Series(
            np.isinf(numeric.to_numpy(dtype=float)).sum(axis=0),
            index=numeric.columns
        )
        print("Affected columns:")
        print(counts[counts > 0])

In [ ]:
import numpy as np

datasets = {
    "Training": X_train,
    "Known Test": X_test_known,
    "PortScan Test": X_zero_day
}

for name, data in datasets.items():

    values = data.to_numpy(dtype=float)

    inf_count = np.isinf(values).sum()
    nan_count = np.isnan(values).sum()

    print(f"\n{name}")
    print("Infinite values:", inf_count)
    print("Missing values:", nan_count)

    assert inf_count == 0, f"Infinite values found in {name}"

print("\nAll datasets are free from infinite values.")

In [ ]:
# STEP 12: PREPROCESSING AND VALIDATION SPLIT

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

RANDOM_STATE = 42

print("=" * 70)
print("PREPROCESSING AND VALIDATION")
print("=" * 70)

# 1. Split training data into training and validation
X_train_core, X_val, y_train_core, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_train
)

# 2. Fit imputer using training data only
imputer = SimpleImputer(
    strategy="median",
    keep_empty_features=True
)

X_train_processed = pd.DataFrame(
    imputer.fit_transform(X_train_core),
    columns=X_train_core.columns,
    index=X_train_core.index
)

X_val_processed = pd.DataFrame(
    imputer.transform(X_val),
    columns=X_train_core.columns,
    index=X_val.index
)

X_test_known_processed = pd.DataFrame(
    imputer.transform(X_test_known),
    columns=X_train_core.columns,
    index=X_test_known.index
)

X_zero_day_processed = pd.DataFrame(
    imputer.transform(X_zero_day),
    columns=X_train_core.columns,
    index=X_zero_day.index
)

# 3. Combine final test data
X_test_processed = pd.concat(
    [
        X_test_known_processed,
        X_zero_day_processed
    ],
    axis=0
)

# 4. Prepare Isolation Forest data
# Only benign training records

X_if_train_processed = X_train_processed.loc[
    y_train_core == 0
].copy()

# Scale anomaly detection features

if_scaler = RobustScaler()

X_if_train_scaled = if_scaler.fit_transform(
    X_if_train_processed
)

X_val_scaled = if_scaler.transform(
    X_val_processed
)

X_test_scaled = if_scaler.transform(
    X_test_processed
)

# 5. Validation checks
print("\nPREPROCESSING SUMMARY")

print("Training records:", len(X_train_processed))
print("Validation records:", len(X_val_processed))
print("Known testing records:", len(X_test_known_processed))
print("Zero-day testing records:", len(X_zero_day_processed))

print("Total features:", X_train_processed.shape[1])

print(
    "\nMissing values after imputation:",
    X_train_processed.isna().sum().sum()
)

print(
    "Infinite values in training:",
    np.isinf(X_train_processed.to_numpy()).sum()
)

print(
    "Isolation Forest training records:",
    len(X_if_train_scaled)
)

assert np.isfinite(X_train_processed.to_numpy()).all()
assert np.isfinite(X_val_processed.to_numpy()).all()
assert np.isfinite(X_test_processed.to_numpy()).all()

assert len(X_train_processed) == len(y_train_core)

print("\nSUCCESS: Preprocessing completed.")
print("SUCCESS: No missing or infinite values.")
print("SUCCESS: Validation dataset prepared.")
print("SUCCESS: Test data not used for preprocessing fit.")

In [ ]:
# STEP 13: RANDOM FOREST CLASSIFIER

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

import time

print("=" * 70)
print("RANDOM FOREST MODEL TRAINING")
print("=" * 70)

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=25,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

start_time = time.time()

rf_model.fit(
    X_train_processed,
    y_train_core
)

rf_training_time = time.time() - start_time

# Validation predictions

rf_val_predictions = rf_model.predict(
    X_val_processed
)

rf_val_probabilities = rf_model.predict_proba(
    X_val_processed
)[:, 1]

print("\nRANDOM FOREST TRAINING COMPLETED")

print("Training time:",
      round(rf_training_time, 2), "seconds")

print("\nVALIDATION PERFORMANCE")

print("Accuracy:",
      round(accuracy_score(y_val, rf_val_predictions), 4))

print("Precision:",
      round(precision_score(y_val, rf_val_predictions), 4))

print("Recall:",
      round(recall_score(y_val, rf_val_predictions), 4))

print("F1-score:",
      round(f1_score(y_val, rf_val_predictions), 4))

print("\nClassification Report:")

print(
    classification_report(
        y_val,
        rf_val_predictions,
        target_names=["BENIGN", "KNOWN ATTACK"],
        zero_division=0
    )
)

In [ ]:
# STEP 14: XGBOOST TRAINING AND EVALUATION

import time
import numpy as np
import pandas as pd
import xgboost as xgb

from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("=" * 65)
print("XGBOOST MODEL TRAINING")
print("=" * 65)

# Convert data to NumPy arrays
X_train_xgb = np.asarray(X_train_processed, dtype=np.float32)
X_val_xgb = np.asarray(X_val_processed, dtype=np.float32)

y_train_xgb = np.asarray(y_train_core, dtype=np.int32)
y_val_xgb = np.asarray(y_val, dtype=np.int32)

# Handle class imbalance
negatives = np.sum(y_train_xgb == 0)
positives = np.sum(y_train_xgb == 1)

scale_weight = negatives / positives

# Initialize model
xgb_model = XGBClassifier(
    n_estimators=400,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    reg_lambda=2,
    scale_pos_weight=scale_weight,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

# Train model
start_time = time.time()

xgb_model.fit(
    X_train_xgb,
    y_train_xgb,
    verbose=False
)

xgb_training_time = time.time() - start_time

# Generate validation probabilities
xgb_val_probabilities = xgb_model.predict_proba(
    X_val_xgb
)[:, 1]

# Select threshold with FPR <= 1%
threshold_results = []

for threshold in np.arange(0.10, 0.951, 0.01):

    predictions = (
        xgb_val_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val_xgb,
        predictions,
        labels=[0, 1]
    ).ravel()

    fpr = fp / (fp + tn) if (fp + tn) else 0

    threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_val_xgb, predictions),
        "Precision": precision_score(
            y_val_xgb, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val_xgb, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_val_xgb, predictions, zero_division=0
        ),
        "FPR": fpr
    })

threshold_table = pd.DataFrame(threshold_results)

eligible = threshold_table[
    threshold_table["FPR"] <= 0.01
]

if eligible.empty:
    raise ValueError("No threshold satisfies the 1% FPR limit.")

best_row = eligible.loc[eligible["F1"].idxmax()]

xgb_threshold = float(best_row["Threshold"])

# Final validation predictions
xgb_val = (
    xgb_val_probabilities >= xgb_threshold
).astype(int)

# Evaluation
print("\nXGBOOST TRAINING COMPLETED")
print("Training time:", round(xgb_training_time, 2), "seconds")
print("Selected threshold:", round(xgb_threshold, 2))

print("\nVALIDATION PERFORMANCE")

print("Accuracy:",
      round(accuracy_score(y_val_xgb, xgb_val), 4))

print("Precision:",
      round(precision_score(y_val_xgb, xgb_val, zero_division=0), 4))

print("Recall:",
      round(recall_score(y_val_xgb, xgb_val, zero_division=0), 4))

print("F1-score:",
      round(f1_score(y_val_xgb, xgb_val, zero_division=0), 4))

tn, fp, fn, tp = confusion_matrix(
    y_val_xgb, xgb_val, labels=[0, 1]
).ravel()

xgb_fpr = fp / (fp + tn) if (fp + tn) else 0

print("False Positive Rate:", round(xgb_fpr, 4))

print("\nCLASSIFICATION REPORT")

print(classification_report(
    y_val_xgb,
    xgb_val,
    target_names=["BENIGN", "KNOWN ATTACK"],
    zero_division=0
))

print("\nXGBoost evaluation completed successfully.")

In [ ]:
# STEP 15: ISOLATION FOREST ANOMALY DETECTION

import numpy as np
import pandas as pd
import time

from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("=" * 65)
print("ISOLATION FOREST - BASELINE MODEL")
print("=" * 65)

if_model = IsolationForest(
    n_estimators=300,
    max_samples=512,
    contamination="auto",
    max_features=1.0,
    bootstrap=False,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

start_time = time.time()
if_model.fit(X_if_train_scaled)
if_training_time = time.time() - start_time

print("\nTraining completed.")
print("Training time:", round(if_training_time, 2), "seconds")

# Calculate anomaly scores
val_scores = if_model.score_samples(X_val_scaled)

benign_mask = np.asarray(y_val) == 0
benign_scores = val_scores[benign_mask]

# Evaluate different thresholds
thresholds_to_test = [0.01, 0.02, 0.05, 0.10, 0.15]
results = []

for target_fpr in thresholds_to_test:

    threshold = np.quantile(benign_scores, target_fpr)
    predictions = (val_scores < threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val, predictions, labels=[0, 1]
    ).ravel()

    actual_fpr = fp / (fp + tn)

    results.append({
        "Target FPR": target_fpr,
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_val, predictions),
        "Precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "F1 Score": f1_score(
            y_val, predictions, zero_division=0
        ),
        "Actual FPR": actual_fpr
    })

if_results = pd.DataFrame(results)

print("\nTHRESHOLD COMPARISON")
display(if_results.round(4))

# Select the best F1-score under a 5% FPR limit
eligible = if_results[
    if_results["Actual FPR"] <= 0.0501
]

if eligible.empty:
    raise ValueError("No threshold satisfies the FPR limit.")

best_row = eligible.loc[eligible["F1 Score"].idxmax()]

anomaly_threshold = float(best_row["Threshold"])

# Final validation predictions
if_val_predictions = (
    val_scores < anomaly_threshold
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_val, if_val_predictions, labels=[0, 1]
).ravel()

if_fpr = fp / (fp + tn)

print("\nSELECTED MODEL PERFORMANCE")
print("Threshold:", round(anomaly_threshold, 6))
print("Accuracy:", round(
    accuracy_score(y_val, if_val_predictions), 4
))
print("Precision:", round(
    precision_score(y_val, if_val_predictions, zero_division=0), 4
))
print("Recall:", round(
    recall_score(y_val, if_val_predictions, zero_division=0), 4
))
print("F1-score:", round(
    f1_score(y_val, if_val_predictions, zero_division=0), 4
))
print("False Positive Rate:", round(if_fpr * 100, 3), "%")

print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_val,
    if_val_predictions,
    target_names=["BENIGN", "ATTACK"],
    zero_division=0
))

print("\nIsolation Forest baseline completed.")

In [ ]:
# STEP 16: ISOLATION FOREST THRESHOLD OPTIMIZATION

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("=" * 70)
print("ISOLATION FOREST THRESHOLD OPTIMIZATION")
print("=" * 70)

# Generate validation anomaly scores
if_val_scores = if_model.score_samples(X_val_scaled)

y_validation = np.asarray(y_val).astype(int)

# Extract scores of benign validation records
benign_mask = y_validation == 0
benign_val_scores = if_val_scores[benign_mask]

# Evaluate different thresholds
target_fprs = [0.01, 0.02, 0.03, 0.04, 0.05,
               0.075, 0.10, 0.15]

threshold_results = []

for target_fpr in target_fprs:

    threshold = np.quantile(
        benign_val_scores,
        target_fpr
    )

    predictions = (
        if_val_scores < threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_validation,
        predictions,
        labels=[0, 1]
    ).ravel()

    actual_fpr = fp / (fp + tn)

    threshold_results.append({
        "Target FPR": target_fpr,
        "Threshold": threshold,
        "Accuracy": accuracy_score(
            y_validation, predictions
        ),
        "Precision": precision_score(
            y_validation, predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_validation, predictions,
            zero_division=0
        ),
        "F1 Score": f1_score(
            y_validation, predictions,
            zero_division=0
        ),
        "Actual FPR": actual_fpr
    })

threshold_results_df = pd.DataFrame(threshold_results)

print("\nTHRESHOLD PERFORMANCE")
display(threshold_results_df.round(4))

# Select the best F1-score with FPR at or below 5%
eligible = threshold_results_df[
    threshold_results_df["Actual FPR"] <= 0.05
]

if eligible.empty:
    raise ValueError("No threshold satisfies the FPR limit.")

best_result = eligible.loc[
    eligible["F1 Score"].idxmax()
]

optimized_threshold = float(best_result["Threshold"])

# Save final predictions and threshold
if_val_predictions = (
    if_val_scores < optimized_threshold
).astype(int)

print("\nOPTIMIZED ISOLATION FOREST")

print("Selected threshold:",
      round(optimized_threshold, 6))

print("Accuracy:",
      round(accuracy_score(
          y_validation, if_val_predictions
      ), 4))

print("Precision:",
      round(precision_score(
          y_validation, if_val_predictions,
          zero_division=0
      ), 4))

print("Recall:",
      round(recall_score(
          y_validation, if_val_predictions,
          zero_division=0
      ), 4))

print("F1-score:",
      round(f1_score(
          y_validation, if_val_predictions,
          zero_division=0
      ), 4))

tn, fp, fn, tp = confusion_matrix(
    y_validation,
    if_val_predictions,
    labels=[0, 1]
).ravel()

print("False Positive Rate:",
      round(fp / (fp + tn) * 100, 3), "%")

In [ ]:
# STEP 17: ISOLATION FOREST THRESHOLD ANALYSIS

import matplotlib.pyplot as plt

threshold_df = threshold_results_df.copy()

plt.figure(figsize=(10, 5))

plt.plot(
    threshold_df["Actual FPR"] * 100,
    threshold_df["Recall"] * 100,
    marker="o",
    linewidth=2,
    label="Recall"
)

plt.plot(
    threshold_df["Actual FPR"] * 100,
    threshold_df["Precision"] * 100,
    marker="s",
    linewidth=2,
    label="Precision"
)

plt.xlabel("False Positive Rate (%)")
plt.ylabel("Performance (%)")
plt.title("Isolation Forest Threshold Analysis")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nBest threshold:")
print("Threshold:", round(optimized_threshold, 6))
print("F1-score:", round(
    f1_score(y_val, if_val_predictions), 4
))
print("False Positive Rate:", round(
    best_row["Actual FPR"] * 100, 3
), "%")

In [ ]:
# STEP 18: MODEL AND HYBRID PERFORMANCE COMPARISON

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("=" * 70)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 70)

# Prepare validation data
y_true = np.asarray(y_val).astype(int)

X_val_rf = X_val_processed

X_val_xgb = np.asarray(
    X_val_processed, dtype=np.float32
)

X_val_if = X_val_scaled

# Individual model predictions
rf_val = rf_model.predict(X_val_rf).astype(int)

xgb_val_probabilities = xgb_model.predict_proba(
    X_val_xgb
)[:, 1]

xgb_val = (
    xgb_val_probabilities >= xgb_threshold
).astype(int)

if_val_scores = if_model.score_samples(X_val_if)

if_val = (
    if_val_scores < optimized_threshold
).astype(int)

# Compare individual and combined models
strategies = {
    "Random Forest": rf_val,
    "XGBoost": xgb_val,
    "Isolation Forest": if_val,

    "RF + XGBoost": (
        (rf_val == 1) | (xgb_val == 1)
    ).astype(int),

    "RF + XGBoost + Isolation Forest": (
        (rf_val == 1) |
        (xgb_val == 1) |
        (if_val == 1)
    ).astype(int)
}

# Calculate performance metrics
rows = []

for name, predictions in strategies.items():

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "F1 Score": f1_score(
            y_true, predictions, zero_division=0
        ),
        "FPR": fp / (fp + tn) if (fp + tn) else 0,
        "False Positives": fp,
        "Missed Attacks": fn
    })

comparison_df = pd.DataFrame(rows)

comparison_df = comparison_df.sort_values(
    "F1 Score", ascending=False
).reset_index(drop=True)

print("\nVALIDATION PERFORMANCE")
display(comparison_df.round(4))

# Identify the strongest model
best_model = comparison_df.iloc[0]

print("\nBEST VALIDATION MODEL")
print("Model:", best_model["Model"])
print("F1-score:", round(best_model["F1 Score"], 4))
print("Accuracy:", round(best_model["Accuracy"], 4))
print("False Positive Rate:", round(best_model["FPR"] * 100, 3), "%")

In [ ]:
# STEP 19: MODEL PERFORMANCE VISUALIZATION

import matplotlib.pyplot as plt
import seaborn as sns

# Use the updated comparison results
plot_data = comparison_df.copy()

plt.figure(figsize=(12, 6))

sns.barplot(
    data=plot_data,
    x="Model",
    y="F1 Score"
)

plt.title("Model Performance Comparison", fontsize=15)
plt.xlabel("Machine Learning Models")
plt.ylabel("F1 Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=20, ha="right")
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

# Display the best model
best_model = plot_data.loc[plot_data["F1 Score"].idxmax()]

print("\nBEST PERFORMING MODEL")
print("Model:", best_model["Model"])
print("Accuracy:", round(best_model["Accuracy"], 4))
print("Precision:", round(best_model["Precision"], 4))
print("Recall:", round(best_model["Recall"], 4))
print("F1 Score:", round(best_model["F1 Score"], 4))

In [ ]:
# STEP 20: UNSEEN PORTSCAN ATTACK DETECTION

import numpy as np
import pandas as pd

from sklearn.metrics import confusion_matrix

print("=" * 75)
print("ZERO-DAY / UNSEEN ATTACK DETECTION")
print("=" * 75)

# 1. Validate test data

n_known = len(X_test_known_processed)
n_zero_day = len(X_zero_day_processed)
n_total = len(X_test_processed)

assert n_known + n_zero_day == n_total, (
    "Test dataset sizes do not match."
)

assert len(y_test_known) == n_known, (
    "Known test labels do not match test records."
)

assert n_zero_day > 0, "No PortScan records found."

print("\nTEST DATA VALIDATION")
print("Known test records:", n_known)
print("Unseen PortScan records:", n_zero_day)
print("Total test records:", n_total)


# 2. Generate predictions

X_test_rf = X_test_processed

X_test_xgb = np.asarray(
    X_test_processed,
    dtype=np.float32
)

X_test_if = X_test_scaled

# Random Forest
rf_test = rf_model.predict(
    X_test_rf
).astype(int)

# XGBoost using selected validation threshold
xgb_test_probabilities = xgb_model.predict_proba(
    X_test_xgb
)[:, 1]

xgb_test = (
    xgb_test_probabilities >= xgb_threshold
).astype(int)

# Isolation Forest
if_test_scores = if_model.score_samples(
    X_test_if
)

if_test = (
    if_test_scores < optimized_threshold
).astype(int)

# 3. Hybrid predictions

hybrid_or_test = (
    (rf_test == 1) | (xgb_test == 1)
).astype(int)

hybrid_and_test = (
    (rf_test == 1) & (xgb_test == 1)
).astype(int)

hybrid_three_test = (
    (rf_test == 1) |
    (xgb_test == 1) |
    (if_test == 1)
).astype(int)

# 4. Evaluate unseen PortScan attacks
zero_day_results = {
    "Random Forest": rf_test[n_known:],
    "XGBoost": xgb_test[n_known:],
    "RF + XGBoost (OR)": hybrid_or_test[n_known:],
    "RF + XGBoost (AND)": hybrid_and_test[n_known:],
    "Isolation Forest": if_test[n_known:],
    "RF + XGBoost + IF": hybrid_three_test[n_known:]
}

results = []

for name, predictions in zero_day_results.items():

    detected = int(np.sum(predictions == 1))
    missed = int(np.sum(predictions == 0))

    results.append({
        "Model": name,
        "Total Attacks": n_zero_day,
        "Detected": detected,
        "Missed": missed,
        "Detection Rate (%)": detected / n_zero_day * 100
    })

zero_day_comparison = pd.DataFrame(results)

print("\nUNSEEN PORTSCAN DETECTION RESULTS")
display(zero_day_comparison.round(3))

# 5. Evaluate false positives on known benign traffic
known_benign_mask = (
    np.asarray(y_test_known).astype(int) == 0
)

benign_predictions = {
    "Random Forest": rf_test[:n_known],
    "XGBoost": xgb_test[:n_known],
    "RF + XGBoost (OR)": hybrid_or_test[:n_known],
    "RF + XGBoost (AND)": hybrid_and_test[:n_known],
    "Isolation Forest": if_test[:n_known],
    "RF + XGBoost + IF": hybrid_three_test[:n_known]
}

benign_results = []

for name, predictions in benign_predictions.items():

    predictions = predictions[known_benign_mask]

    false_positives = int(
        np.sum(predictions == 1)
    )

    total_benign = len(predictions)

    benign_results.append({
        "Model": name,
        "Benign Records": total_benign,
        "False Positives": false_positives,
        "False Positive Rate (%)": (
            false_positives / total_benign * 100
        )
    })

benign_comparison = pd.DataFrame(benign_results)

print("\nFALSE POSITIVES ON KNOWN BENIGN TRAFFIC")
display(benign_comparison.round(3))

# 6. Summary
best_zero_day = zero_day_comparison.loc[
    zero_day_comparison["Detection Rate (%)"].idxmax()
]

print("\nBEST PORTSCAN DETECTION")
print("Model:", best_zero_day["Model"])
print("Detection rate:",
      round(best_zero_day["Detection Rate (%)"], 3), "%")

print("\nSelected XGBoost threshold:", xgb_threshold)
print("Selected Isolation Forest threshold:", optimized_threshold)

In [ ]:
# STEP 21: ISOLATION FOREST ANOMALY SCORE ANALYSIS

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 65)
print("ISOLATION FOREST SCORE DISTRIBUTION")
print("=" * 65)

# X_test_scaled already contains:
# 1. Known test records first
# 2. Unseen PortScan records afterwards

n_known = len(X_test_known_processed)

# Calculate scores using already-scaled data
known_scores = if_model.score_samples(
    X_test_scaled[:n_known]
)

portscan_scores = if_model.score_samples(
    X_test_scaled[n_known:]
)

# Separate known benign and known attack scores
benign_mask = (y_test_known == 0)

benign_scores = known_scores[benign_mask]
known_attack_scores = known_scores[~benign_mask]

# Score summary
score_summary = pd.DataFrame({
    "Traffic Category": [
        "Known Benign",
        "Known Attacks",
        "Unseen PortScan"
    ],
    "Mean Score": [
        np.mean(benign_scores),
        np.mean(known_attack_scores),
        np.mean(portscan_scores)
    ],
    "Median Score": [
        np.median(benign_scores),
        np.median(known_attack_scores),
        np.median(portscan_scores)
    ],
    "Minimum Score": [
        np.min(benign_scores),
        np.min(known_attack_scores),
        np.min(portscan_scores)
    ],
    "Maximum Score": [
        np.max(benign_scores),
        np.max(known_attack_scores),
        np.max(portscan_scores)
    ]
})

display(score_summary.round(5))

# Plot score distributions
plt.figure(figsize=(12, 6))

plt.hist(
    benign_scores,
    bins=60,
    alpha=0.55,
    density=True,
    label="Known Benign"
)

plt.hist(
    known_attack_scores,
    bins=60,
    alpha=0.55,
    density=True,
    label="Known Attacks"
)

plt.hist(
    portscan_scores,
    bins=60,
    alpha=0.55,
    density=True,
    label="Unseen PortScan"
)

plt.axvline(
    optimized_threshold,
    linestyle="--",
    linewidth=2,
    label="Selected IF Threshold"
)

plt.xlabel("Isolation Forest Score")
plt.ylabel("Density")
plt.title("Isolation Forest Anomaly Score Distribution")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

print("\nAnomaly score analysis completed.")

In [ ]:
# STEP 22: PORTSCAN VS BENIGN FEATURE ANALYSIS

import numpy as np
import pandas as pd

print("=" * 70)
print("PORTSCAN VS BENIGN FEATURE ANALYSIS")
print("=" * 70)

# Separate benign and unseen PortScan records
benign_features = X_test_known_processed.loc[
    np.asarray(y_test_known) == 0
].copy()

portscan_features = X_zero_day_processed.copy()

# Ensure feature columns match
common_features = [
    col for col in benign_features.columns
    if col in portscan_features.columns
]

benign_features = benign_features[common_features]
portscan_features = portscan_features[common_features]

# Calculate medians and interquartile ranges
benign_median = benign_features.median()
portscan_median = portscan_features.median()

benign_iqr = (
    benign_features.quantile(0.75) -
    benign_features.quantile(0.25)
)

# Avoid division by zero
benign_iqr = benign_iqr.replace(0, np.nan)

# Calculate standardized median differences
comparison = pd.DataFrame({
    "Feature": common_features,
    "Benign Median": benign_median.values,
    "PortScan Median": portscan_median.values,
    "Absolute Difference": (
        portscan_median - benign_median
    ).abs().values,
    "Standardized Difference": (
        (portscan_median - benign_median) / benign_iqr
    ).abs().values
})

comparison = comparison.replace(
    [np.inf, -np.inf], np.nan
)

comparison = comparison.dropna(
    subset=["Standardized Difference"]
)

comparison = comparison.sort_values(
    "Standardized Difference",
    ascending=False
)

print("\nTOP 20 FEATURES BY STANDARDIZED DIFFERENCE")

display(comparison.head(20).round(4))

# Identify potentially relevant network features
scan_features = [
    col for col in common_features
    if any(term in col.upper() for term in [
        "PORT", "SYN", "RST", "FLOW",
        "PACKET", "FWD", "BWD", "IAT"
    ])
]

print("\nAVAILABLE NETWORK TRAFFIC FEATURES")
print(scan_features)

print("\nTotal common features:", len(common_features))

In [ ]:
# STEP 23: FEATURE DISTRIBUTION ANALYSIS

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("=" * 65)
print("FEATURE DISTRIBUTION ANALYSIS")
print("=" * 65)

selected_features = [
    "FLOW BYTES/S",
    "FLOW PACKETS/S",
    "FLOW DURATION",
    "FLOW IAT MAX",
    "FWD PACKETS/S",
    "BWD PACKETS/S",
    "DESTINATION PORT",
    "PACKET LENGTH VARIANCE"
]

# Separate benign and PortScan records
benign_data = X_test_known_processed.loc[
    np.asarray(y_test_known) == 0,
    selected_features
].copy()

portscan_data = X_zero_day_processed[
    selected_features
].copy()

# Take samples for visualization
benign_sample = benign_data.sample(
    n=min(5000, len(benign_data)),
    random_state=42
)

portscan_sample = portscan_data.sample(
    n=min(5000, len(portscan_data)),
    random_state=42
)

# Display distribution statistics
distribution_summary = pd.DataFrame({
    "Feature": selected_features,
    "Benign Median": benign_data.median().values,
    "PortScan Median": portscan_data.median().values,
    "Benign Std": benign_data.std().values,
    "PortScan Std": portscan_data.std().values
})

print("\nFEATURE STATISTICS")
display(distribution_summary.round(3))

# Plot feature distributions
fig, axes = plt.subplots(4, 2, figsize=(15, 16))

for i, feature in enumerate(selected_features):

    ax = axes[i // 2, i % 2]

    benign_values = np.log1p(
        benign_sample[feature].clip(lower=0)
    )

    portscan_values = np.log1p(
        portscan_sample[feature].clip(lower=0)
    )

    ax.hist(
        benign_values,
        bins=40,
        alpha=0.6,
        density=True,
        label="Benign"
    )

    ax.hist(
        portscan_values,
        bins=40,
        alpha=0.6,
        density=True,
        label="PortScan"
    )

    ax.set_title(feature)
    ax.set_xlabel("Log-transformed value")
    ax.set_ylabel("Density")
    ax.legend()
    ax.grid(alpha=0.2)

plt.suptitle(
    "Network Traffic Feature Distributions",
    fontsize=15
)

plt.tight_layout()
plt.show()

In [ ]:
# STEP 24: FEATURE SELECTION USING TRAINING DATA ONLY

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier

print("=" * 70)
print("TRAINING-BASED FEATURE SELECTION")
print("=" * 70)

# Train feature selector using training data only
feature_selector = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

feature_selector.fit(
    np.asarray(X_train_processed, dtype=np.float32),
    np.asarray(y_train_core).astype(int)
)

# Calculate feature importance
importance_df = pd.DataFrame({
    "Feature": X_train_processed.columns,
    "Importance": feature_selector.feature_importances_
})

importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

importance_df["Cumulative Importance"] = (
    importance_df["Importance"].cumsum()
)

print("\nTOP 30 IMPORTANT FEATURES")
display(importance_df.head(30).round(5))

# Select top 20 features
selected_feature_names = (
    importance_df.head(20)["Feature"].tolist()
)

print("\nSELECTED 20 FEATURES")
for i, feature in enumerate(selected_feature_names, 1):
    print(f"{i:2}. {feature}")

print(
    "\nCumulative importance of selected features:",
    round(
        importance_df.head(20)["Importance"].sum() * 100,
        2
    ),
    "%"
)

# Visualize feature importance
top_features = importance_df.head(20).sort_values(
    "Importance",
    ascending=True
)

plt.figure(figsize=(11, 8))

plt.barh(
    top_features["Feature"],
    top_features["Importance"]
)

plt.xlabel("Feature Importance")
plt.ylabel("Network Features")
plt.title("Top 20 Features from Training Data")
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# STEP 25: IMPROVED ISOLATION FOREST

import numpy as np
import pandas as pd
import time

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("=" * 70)
print("IMPROVED ISOLATION FOREST")
print("=" * 70)

# Select the 20 important features
X_train_selected = X_train_processed[selected_feature_names].copy()
X_val_selected = X_val_processed[selected_feature_names].copy()

# Use benign training records only
benign_train = X_train_selected[
    np.asarray(y_train_core) == 0
]

# Scale using benign training data
selected_scaler = RobustScaler()

X_benign_scaled = selected_scaler.fit_transform(
    benign_train
)

X_val_selected_scaled = selected_scaler.transform(
    X_val_selected
)

# Train Isolation Forest
improved_if_model = IsolationForest(
    n_estimators=300,
    max_samples=512,
    contamination="auto",
    max_features=1.0,
    bootstrap=False,
    n_jobs=-1,
    random_state=42
)

start_time = time.time()

improved_if_model.fit(X_benign_scaled)

training_time = time.time() - start_time

# Calculate validation anomaly scores
val_scores = improved_if_model.score_samples(
    X_val_selected_scaled
)

y_val_array = np.asarray(y_val).astype(int)

# Calibrate threshold using benign validation records
benign_val_scores = val_scores[y_val_array == 0]

threshold_rows = []

for target_fpr in [0.01, 0.02, 0.03, 0.04, 0.05]:

    threshold = np.quantile(
        benign_val_scores,
        target_fpr
    )

    predictions = (val_scores < threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val_array,
        predictions,
        labels=[0, 1]
    ).ravel()

    actual_fpr = fp / (fp + tn) if (fp + tn) else 0

    threshold_rows.append({
        "Target FPR": target_fpr,
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_val_array, predictions),
        "Precision": precision_score(
            y_val_array, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val_array, predictions, zero_division=0
        ),
        "F1 Score": f1_score(
            y_val_array, predictions, zero_division=0
        ),
        "Actual FPR": actual_fpr
    })

threshold_results = pd.DataFrame(threshold_rows)

# Choose the highest F1 while keeping FPR at or below 5%
eligible = threshold_results[
    threshold_results["Actual FPR"] <= 0.05
]

best_row = eligible.loc[
    eligible["F1 Score"].idxmax()
]

improved_if_threshold = float(best_row["Threshold"])

improved_val_predictions = (
    val_scores < improved_if_threshold
).astype(int)

print(f"\nTraining time: {training_time:.2f} seconds")

print("\nTHRESHOLD COMPARISON")
display(threshold_results.round(4))

print("\nIMPROVED ISOLATION FOREST PERFORMANCE")
print(f"Selected threshold: {improved_if_threshold:.6f}")
print(f"Accuracy: {accuracy_score(y_val_array, improved_val_predictions):.4f}")
print(f"Precision: {precision_score(y_val_array, improved_val_predictions, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_val_array, improved_val_predictions, zero_division=0):.4f}")
print(f"F1-score: {f1_score(y_val_array, improved_val_predictions, zero_division=0):.4f}")

tn, fp, fn, tp = confusion_matrix(
    y_val_array,
    improved_val_predictions,
    labels=[0, 1]
).ravel()

print(f"False Positive Rate: {fp / (fp + tn) * 100:.3f}%")
print(f"False Positives: {fp}")
print(f"Missed Attacks: {fn}")

In [ ]:
# STEP 26: ORIGINAL VS IMPROVED ISOLATION FOREST

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("=" * 70)
print("ORIGINAL VS IMPROVED ISOLATION FOREST")
print("=" * 70)

y_true = np.asarray(y_val).astype(int)

# Original Isolation Forest
original_predictions = (
    if_val_scores < optimized_threshold
).astype(int)

# Improved Isolation Forest
improved_predictions = (
    val_scores < improved_if_threshold
).astype(int)

models = {
    "Original IF (70 Features)": original_predictions,
    "Improved IF (20 Features)": improved_predictions
}

results = []

for name, predictions in models.items():

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(y_true, predictions, zero_division=0),
        "Recall": recall_score(y_true, predictions, zero_division=0),
        "F1 Score": f1_score(y_true, predictions, zero_division=0),
        "False Positive Rate": fp / (fp + tn),
        "True Positives": tp,
        "False Positives": fp,
        "Missed Attacks": fn,
        "True Negatives": tn
    })

if_comparison = pd.DataFrame(results)

display(if_comparison.round(4))

print("\nISOLATION FOREST COMPARISON COMPLETED.")

In [ ]:
# STEP 27: CHECK THE IMPROVED HYBRID OPTIONS

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("=" * 70)
print("IMPROVED HYBRID MODEL COMPARISON")
print("=" * 70)

y_true = np.asarray(y_val).astype(int)

# Random Forest predictions
rf_val = rf_model.predict(X_val_processed)

# XGBoost predictions
xgb_val = (
    xgb_model.predict_proba(X_val_processed)[:, 1] >= xgb_threshold
).astype(int)

# Improved Isolation Forest predictions
improved_if_val = (
    val_scores < improved_if_threshold
).astype(int)

# Compare different combinations
options = {
    "Random Forest": rf_val,
    "XGBoost": xgb_val,

    "RF + XGBoost": (
        (rf_val == 1) | (xgb_val == 1)
    ).astype(int),

    "RF + XGBoost + Improved IF": (
        (rf_val == 1) |
        (xgb_val == 1) |
        (improved_if_val == 1)
    ).astype(int),

    "XGBoost + Improved IF": (
        (xgb_val == 1) |
        (improved_if_val == 1)
    ).astype(int)
}

rows = []

for name, predictions in options.items():

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "F1 Score": f1_score(
            y_true, predictions, zero_division=0
        ),
        "FPR": fp / (fp + tn) if (fp + tn) else 0,
        "True Positives": tp,
        "False Positives": fp,
        "Missed Attacks": fn,
        "True Negatives": tn
    })

improved_hybrid_results = pd.DataFrame(rows)

improved_hybrid_results = improved_hybrid_results.sort_values(
    by="F1 Score",
    ascending=False
).reset_index(drop=True)

print("\nVALIDATION PERFORMANCE")
display(improved_hybrid_results.round(4))

best_model = improved_hybrid_results.iloc[0]

print("\nBEST VALIDATION MODEL")
print("Model:", best_model["Model"])
print(f"Accuracy: {best_model['Accuracy']:.4f}")
print(f"Precision: {best_model['Precision']:.4f}")
print(f"Recall: {best_model['Recall']:.4f}")
print(f"F1 Score: {best_model['F1 Score']:.4f}")
print(f"False Positive Rate: {best_model['FPR']:.4f}")

In [ ]:
# STEP 28: TWO-STAGE HYBRID CYBERATTACK DETECTION

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)

print("=" * 75)
print("TWO-STAGE HYBRID CYBERATTACK DETECTION")
print("=" * 75)

# STAGE 1: XGBOOST SUPERVISED DETECTION
xgb_predictions = (
    xgb_model.predict_proba(X_val_processed)[:, 1] >= xgb_threshold
).astype(int)

# STAGE 2: IMPROVED ISOLATION FOREST ANOMALY DETECTION
anomaly_scores = improved_if_model.score_samples(
    X_val_selected_scaled
)

anomaly_predictions = (
    anomaly_scores < improved_if_threshold
).astype(int)

# FINAL THREE-CATEGORY CLASSIFICATION
detection_status = np.where(
    xgb_predictions == 1,
    "Known Attack Detected",
    np.where(
        anomaly_predictions == 1,
        "Suspicious Anomaly",
        "Normal Traffic"
    )
)

# DETECTION SUMMARY

status_order = [
    "Normal Traffic",
    "Known Attack Detected",
    "Suspicious Anomaly"
]

status_counts = (
    pd.Series(detection_status)
    .value_counts()
    .reindex(status_order, fill_value=0)
)

print("\nDETECTION SUMMARY")

display(
    status_counts.rename_axis("Detection Status")
    .reset_index(name="Number of Records")
)

# SUPERVISED MODEL PERFORMANCE
print("\nXGBOOST VALIDATION PERFORMANCE")

print("Accuracy:",
      round(accuracy_score(y_val, xgb_predictions), 4))

print("Precision:",
      round(precision_score(y_val, xgb_predictions, zero_division=0), 4))

print("Recall:",
      round(recall_score(y_val, xgb_predictions, zero_division=0), 4))

print("F1 Score:",
      round(f1_score(y_val, xgb_predictions, zero_division=0), 4))

# COMPLETE TWO-STAGE BINARY ALERT PERFORMANCE
final_predictions = (
    (xgb_predictions == 1) |
    (anomaly_predictions == 1)
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_val, final_predictions, labels=[0, 1]
).ravel()

print("\nCOMPLETE TWO-STAGE ALERT PERFORMANCE")

print("Accuracy:", round(accuracy_score(y_val, final_predictions), 4))
print("Precision:", round(precision_score(y_val, final_predictions, zero_division=0), 4))
print("Recall:", round(recall_score(y_val, final_predictions, zero_division=0), 4))
print("F1 Score:", round(f1_score(y_val, final_predictions, zero_division=0), 4))
print("False Positive Rate:", round(fp / (fp + tn), 4))

print("\nTrue Positives:", tp)
print("False Positives:", fp)
print("Missed Attacks:", fn)
print("True Negatives:", tn)

In [ ]:
# STEP 29: MODEL PERFORMANCE COMPARISON
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("=" * 75)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 75)

# GENERATE VALIDATION PREDICTIONS
rf_predictions = rf_model.predict(X_val_processed)

xgb_predictions = (
    xgb_model.predict_proba(X_val_processed)[:, 1] >= xgb_threshold
).astype(int)

# Improved Isolation Forest predictions from STEP 26
if_predictions = improved_if_val

# COMPARE DIFFERENT MODELS
predictions = {
    "Random Forest": rf_predictions,
    "XGBoost": xgb_predictions,

    "RF + XGBoost": (
        (rf_predictions == 1) | (xgb_predictions == 1)
    ).astype(int),

    "Improved Isolation Forest": if_predictions,

    "Two-Stage Hybrid": (
        (xgb_predictions == 1) | (if_predictions == 1)
    ).astype(int)
}

# CALCULATE PERFORMANCE METRICS
rows = []

for name, pred in predictions.items():

    tn, fp, fn, tp = confusion_matrix(
        y_val, pred, labels=[0, 1]
    ).ravel()

    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, pred),
        "Precision": precision_score(y_val, pred, zero_division=0),
        "Recall": recall_score(y_val, pred, zero_division=0),
        "F1 Score": f1_score(y_val, pred, zero_division=0),
        "FPR": fp / (fp + tn) if (fp + tn) else 0,
        "False Positives": fp,
        "Missed Attacks": fn
    })

metrics_df = (
    pd.DataFrame(rows)
    .sort_values("F1 Score", ascending=False)
    .reset_index(drop=True)
)

print("\nVALIDATION PERFORMANCE")

display(metrics_df.round(4))

# PERFORMANCE COMPARISON GRAPH
ax = metrics_df.set_index("Model")[
    ["Accuracy", "Precision", "Recall", "F1 Score"]
].plot(
    kind="bar",
    figsize=(13, 6),
    ylim=(0, 1.05)
)

ax.set_ylabel("Validation Score")
ax.set_title("Machine Learning Model Performance Comparison")
ax.grid(axis="y", alpha=0.25)

plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

# FALSE POSITIVE RATE COMPARISON
ax = metrics_df.set_index("Model")["FPR"].plot(
    kind="bar",
    figsize=(11, 5)
)

ax.set_ylabel("False Positive Rate")
ax.set_title("False Alarm Rate Comparison")
ax.grid(axis="y", alpha=0.25)

plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

# BEST MODEL SUMMARY
best = metrics_df.iloc[0]

print("\nBEST VALIDATION MODEL")
print("Model:", best["Model"])
print("Accuracy:", round(best["Accuracy"], 4))
print("Precision:", round(best["Precision"], 4))
print("Recall:", round(best["Recall"], 4))
print("F1 Score:", round(best["F1 Score"], 4))
print("False Positive Rate:", round(best["FPR"], 4))

In [ ]:
# STEP 30: CONFUSION MATRIX VISUALIZATION
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from sklearn.metrics import confusion_matrix

print("=" * 70)
print("CONFUSION MATRIX COMPARISON")
print("=" * 70)

# GENERATE PREDICTIONS
rf_predictions = rf_model.predict(X_val_processed)

xgb_predictions = (
    xgb_model.predict_proba(X_val_processed)[:, 1] >= xgb_threshold
).astype(int)

hybrid_predictions = (
    (rf_predictions == 1) |
    (xgb_predictions == 1)
).astype(int)

# MODEL COMPARISON
models = {
    "Random Forest": rf_predictions,
    "XGBoost": xgb_predictions,
    "RF + XGBoost (OR)": hybrid_predictions
}

# GENERATE CONFUSION MATRICES
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (model_name, predictions) in zip(axes, models.items()):

    cm = confusion_matrix(
        y_val,
        predictions,
        labels=[0, 1]
    )

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Benign", "Attack"],
        yticklabels=["Benign", "Attack"],
        ax=ax
    )

    ax.set_title(model_name)
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("Actual Label")

plt.suptitle(
    "Confusion Matrix Comparison",
    fontsize=16
)

plt.tight_layout()
plt.show()

# DISPLAY CONFUSION MATRIX VALUES

for name, predictions in models.items():

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        predictions,
        labels=[0, 1]
    ).ravel()

    print(f"\n{name}")
    print(f"True Negatives: {tn}")
    print(f"False Positives: {fp}")
    print(f"False Negatives: {fn}")
    print(f"True Positives: {tp}")

In [ ]:
# STEP 31: ROC CURVE AND AUC COMPARISON

import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    auc
)

# Get prediction probabilities
rf_probabilities = rf_model.predict_proba(
    X_val_processed
)[:, 1]

xgb_probabilities = xgb_model.predict_proba(
    X_val_processed
)[:, 1]

# Calculate ROC curves
rf_fpr, rf_tpr, _ = roc_curve(
    y_val,
    rf_probabilities
)

xgb_fpr, xgb_tpr, _ = roc_curve(
    y_val,
    xgb_probabilities
)

# Calculate AUC
rf_auc = auc(rf_fpr, rf_tpr)
xgb_auc = auc(xgb_fpr, xgb_tpr)

# Plot ROC curves
plt.figure(figsize=(10, 7))

plt.plot(
    rf_fpr,
    rf_tpr,
    label=f"Random Forest (AUC = {rf_auc:.4f})",
    linewidth=2
)

plt.plot(
    xgb_fpr,
    xgb_tpr,
    label=f"XGBoost (AUC = {xgb_auc:.4f})",
    linewidth=2
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("ROC Curve - Cyberattack Detection")

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Random Forest ROC-AUC:", round(rf_auc, 5))
print("XGBoost ROC-AUC:", round(xgb_auc, 5))

print("\nROC analysis completed.")

In [ ]:
# STEP 32: PRECISION-RECALL CURVE
import matplotlib.pyplot as plt

from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score
)

# Calculate precision-recall values
rf_precision, rf_recall, _ = precision_recall_curve(
    y_val,
    rf_probabilities
)

xgb_precision, xgb_recall, _ = precision_recall_curve(
    y_val,
    xgb_probabilities
)

# Calculate Average Precision
rf_ap = average_precision_score(
    y_val,
    rf_probabilities
)

xgb_ap = average_precision_score(
    y_val,
    xgb_probabilities
)

# Plot
plt.figure(figsize=(10, 7))

plt.plot(
    rf_recall,
    rf_precision,
    label=f"Random Forest (AP = {rf_ap:.5f})",
    linewidth=2
)

plt.plot(
    xgb_recall,
    xgb_precision,
    label=f"XGBoost (AP = {xgb_ap:.5f})",
    linewidth=2
)

plt.xlabel("Recall")
plt.ylabel("Precision")

plt.title("Precision-Recall Curve - Cyberattack Detection")

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Random Forest Average Precision:", round(rf_ap, 5))
print("XGBoost Average Precision:", round(xgb_ap, 5))

print("\nPrecision-Recall analysis completed.")

In [ ]:
# STEP 33: SAVE TRAINED MODELS AND PREPROCESSING OBJECTS
import os
import joblib
import json

MODEL_DIR = str(PROJECT_DIR / "output" / "saved_models")

os.makedirs(MODEL_DIR, exist_ok=True)

# Save supervised models
joblib.dump(rf_model, f"{MODEL_DIR}/random_forest.pkl")
joblib.dump(xgb_model, f"{MODEL_DIR}/xgboost.pkl")

# Save improved Isolation Forest
joblib.dump(
    improved_if_model,
    f"{MODEL_DIR}/improved_isolation_forest.pkl"
)

# Save preprocessing objects
joblib.dump(imputer, f"{MODEL_DIR}/median_imputer.pkl")
joblib.dump(selected_scaler, f"{MODEL_DIR}/selected_feature_scaler.pkl")

# Save metadata
metadata = {
    "selected_features": selected_feature_names,
    "original_features": list(rf_model.feature_names_in_),
    "isolation_forest_threshold": float(improved_if_threshold),
    "xgboost_threshold": float(xgb_threshold),
    "supervised_label_mapping": {
        "BENIGN": 0,
        "ATTACK": 1
    },
    "number_of_original_features": len(rf_model.feature_names_in_),
    "number_of_selected_features": len(selected_feature_names),
    "random_state": 42
}

with open(f"{MODEL_DIR}/model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("=" * 65)
print("MODEL SAVING COMPLETED")
print("=" * 65)

for file in sorted(os.listdir(MODEL_DIR)):
    print("Saved:", file)

In [ ]:
# STEP 34: LOAD AND VERIFY SAVED MODELS

import joblib
import os
import json

MODEL_DIR = str(PROJECT_DIR / "output" / "saved_models")

# Load models
loaded_rf = joblib.load(
    f"{MODEL_DIR}/random_forest.pkl"
)

loaded_xgb = joblib.load(
    f"{MODEL_DIR}/xgboost.pkl"
)

loaded_if = joblib.load(
    f"{MODEL_DIR}/improved_isolation_forest.pkl"
)

# Load preprocessing objects
loaded_imputer = joblib.load(
    f"{MODEL_DIR}/median_imputer.pkl"
)

loaded_scaler = joblib.load(
    f"{MODEL_DIR}/selected_feature_scaler.pkl"
)

# Load metadata
with open(
    f"{MODEL_DIR}/model_metadata.json",
    "r"
) as f:
    loaded_metadata = json.load(f)

print("=" * 65)
print("MODEL LOADING VERIFICATION")
print("=" * 65)

print("Random Forest:", type(loaded_rf).__name__)
print("XGBoost:", type(loaded_xgb).__name__)
print("Isolation Forest:", type(loaded_if).__name__)

print("Imputer:", type(loaded_imputer).__name__)
print("Scaler:", type(loaded_scaler).__name__)

print(
    "Selected features:",
    len(loaded_metadata["selected_features"])
)

print(
    "Isolation Forest threshold:",
    loaded_metadata["isolation_forest_threshold"]
)

print("\nAll saved objects loaded successfully.")

In [ ]:
# STEP 35: CYBERATTACK DETECTION PREDICTION FUNCTION
import numpy as np
import pandas as pd

selected_features = loaded_metadata["selected_features"]

if_threshold = loaded_metadata["isolation_forest_threshold"]
xgb_threshold = loaded_metadata["xgboost_threshold"]

original_features = list(loaded_rf.feature_names_in_)


def detect_cyberattack(network_record):

    if isinstance(network_record, pd.Series):
        network_record = network_record.to_frame().T

    if not isinstance(network_record, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame or Series.")

    missing_features = [
        col for col in original_features
        if col not in network_record.columns
    ]

    if missing_features:
        raise ValueError(
            f"Missing required features: {missing_features}"
        )

    X = network_record[original_features].copy()

    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.mask(X < 0, np.nan)

    X_imputed = loaded_imputer.transform(X)

    X_imputed = pd.DataFrame(
        X_imputed,
        columns=original_features,
        index=X.index
    )

    # Stage 1: XGBoost classification

    xgb_probability = loaded_xgb.predict_proba(
        X_imputed
    )[:, 1]

    xgb_prediction = (
        xgb_probability >= xgb_threshold
    ).astype(int)

    # Stage 2: Isolation Forest anomaly detection

    X_selected = X_imputed[selected_features]

    X_selected_scaled = loaded_scaler.transform(
        X_selected
    )

    anomaly_scores = loaded_if.score_samples(
        X_selected_scaled
    )

    anomaly_prediction = (
        anomaly_scores < if_threshold
    ).astype(int)

    # Final classification

    results = []

    for i in range(len(X)):

        if xgb_prediction[i] == 1:
            status = "Known Attack Detected"

        elif anomaly_prediction[i] == 1:
            status = "Suspicious Anomaly"

        else:
            status = "Normal Traffic"

        results.append({
            "Detection Status": status,
            "XGBoost Prediction": int(xgb_prediction[i]),
            "XGBoost Attack Probability": round(
                float(xgb_probability[i]), 5
            ),
            "Isolation Forest Score": round(
                float(anomaly_scores[i]), 5
            ),
            "Anomaly Flag": int(anomaly_prediction[i])
        })

    return pd.DataFrame(results)


print("Cyberattack detection function created successfully.")

In [ ]:
# STEP 36: TEST THE CYBERATTACK DETECTION FUNCTION
import numpy as np

print("=" * 70)
print("CYBERATTACK DETECTION DEMONSTRATION")
print("=" * 70)

# Find benign and malicious test records
benign_indices = np.where(
    np.asarray(y_test_known) == 0
)[0]

attack_indices = np.where(
    np.asarray(y_test_known) == 1
)[0]

assert len(benign_indices) > 0, "No benign test records found."
assert len(attack_indices) > 0, "No attack test records found."

# Extract original records
benign_sample = X_test_known.iloc[[benign_indices[0]]]
attack_sample = X_test_known.iloc[[attack_indices[0]]]

# Test 1: Benign traffic
print("\nTEST 1: BENIGN NETWORK TRAFFIC")

benign_result = detect_cyberattack(benign_sample)
display(benign_result)

# Test 2: Known malicious traffic
print("\nTEST 2: KNOWN MALICIOUS NETWORK TRAFFIC")

attack_result = detect_cyberattack(attack_sample)
display(attack_result)

In [ ]:
# STEP 37: UNSEEN PORTSCAN DEMONSTRATION
# Select one PortScan record
portscan_sample = X_zero_day.iloc[[0]]

print("=" * 70)
print("TEST 3: UNSEEN PORTSCAN TRAFFIC")
print("=" * 70)

portscan_result = detect_cyberattack(
    portscan_sample
)

display(portscan_result)

print("\nActual category: PORTSCAN (withheld from training)")

In [ ]:
# STEP 38: FIND A SUSPICIOUS ANOMALY IN VALIDATION DATA

import numpy as np

print("=" * 70)
print("SEARCHING FOR SUSPICIOUS ANOMALIES")
print("=" * 70)

supervised_val_predictions = np.asarray(xgb_val)
anomaly_val_predictions = (
    np.asarray(val_scores) < improved_if_threshold
).astype(int)

benign_mask = np.asarray(y_val) == 0

suspicious_indices = np.where(
    benign_mask &
    (supervised_val_predictions == 0) &
    (anomaly_val_predictions == 1)
)[0]

print("Suspicious records found:", len(suspicious_indices))

if len(suspicious_indices) > 0:

    idx = suspicious_indices[0]

    print("\nSUSPICIOUS ANOMALY FOUND")
    print("Validation index:", idx)
    print("Actual label: BENIGN")
    print("XGBoost prediction:", supervised_val_predictions[idx])
    print("Isolation Forest score:", val_scores[idx])
    print("Anomaly flag:", anomaly_val_predictions[idx])

    print("\nExpected system output: SUSPICIOUS ANOMALY")
    print("This record is flagged for investigation, not confirmed as an attack.")

else:
    print("\nNo suitable suspicious anomaly was found.")
    print("Do not manually alter the model prediction.")

In [ ]:
# STEP 39: FINAL THREE-CATEGORY DETECTION TEST

print("=" * 75)
print("FINAL THREE-CATEGORY CYBERATTACK DETECTION")
print("=" * 75)

suspicious_idx = suspicious_indices[0]
suspicious_sample = X_val.iloc[[suspicious_idx]]

test_cases = {
    "TEST 1 - NORMAL TRAFFIC": benign_sample,
    "TEST 2 - KNOWN ATTACK": attack_sample,
    "TEST 3 - SUSPICIOUS ANOMALY": suspicious_sample
}

for test_name, sample in test_cases.items():

    print("\n" + "-" * 75)
    print(test_name)
    print("-" * 75)

    result = detect_cyberattack(sample)
    display(result)

print("\n" + "=" * 75)
print("THREE-CATEGORY DEMONSTRATION FINISHED")
print("=" * 75)

In [ ]:
# STEP 40: EVALUATE THE TWO-STAGE DETECTOR

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, ConfusionMatrixDisplay
)

y_true = np.asarray(y_val).astype(int)

# Stage 1: XGBoost known attack detection
xgb_pred = (
    xgb_model.predict_proba(X_val_processed)[:, 1] >= xgb_threshold
).astype(int)

# Stage 2: Improved Isolation Forest anomaly detection
anomaly_pred = (
    np.asarray(val_scores) < improved_if_threshold
).astype(int)

# 0 = Normal, 1 = Known Attack, 2 = Suspicious Anomaly
status = np.zeros(len(y_true), dtype=int)

status[xgb_pred == 1] = 1
status[(xgb_pred == 0) & (anomaly_pred == 1)] = 2

# Any alert is counted as a positive prediction
alarm_pred = (status != 0).astype(int)

def get_metrics(actual, predicted):
    tn, fp, fn, tp = confusion_matrix(
        actual, predicted, labels=[0, 1]
    ).ravel()

    return {
        "Accuracy": accuracy_score(actual, predicted),
        "Precision": precision_score(actual, predicted, zero_division=0),
        "Recall": recall_score(actual, predicted, zero_division=0),
        "F1 Score": f1_score(actual, predicted, zero_division=0),
        "FPR": fp / (fp + tn) if (fp + tn) else 0
    }

comparison = pd.DataFrame([
    {
        "System": "XGBoost",
        **get_metrics(y_true, xgb_pred)
    },
    {
        "System": "Two-Stage Hybrid",
        **get_metrics(y_true, alarm_pred)
    }
])

print("MODEL PERFORMANCE COMPARISON")
display(comparison.round(4))

print("\nDETECTION OUTCOMES")

for code, label in {
    0: "Normal Traffic",
    1: "Known Attack Detected",
    2: "Suspicious Anomaly"
}.items():
    print(f"{label}: {(status == code).sum():,}")

print("\nCONFUSION MATRIX - TWO-STAGE HYBRID")

ConfusionMatrixDisplay.from_predictions(
    y_true,
    alarm_pred,
    labels=[0, 1],
    display_labels=["Benign", "Attack"],
    values_format="d"
)

plt.title("Two-Stage Hybrid Detection")
plt.tight_layout()
plt.show()

print("\nEvaluation completed.")
print("Anomaly alerts indicate unusual traffic, not confirmed attacks.")

In [ ]:
# STEP 41: FINAL CHECK ON HELD-OUT DATA

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)
import numpy as np
import pandas as pd

selected_feature_names = metadata["selected_features"]

# Evaluate known traffic
y_known_test = np.asarray(y_test_known).astype(int)

rf_known = rf_model.predict(X_test_known_processed)

xgb_known_prob = xgb_model.predict_proba(
    X_test_known_processed
)[:, 1]

xgb_known = (xgb_known_prob >= xgb_threshold).astype(int)

known_if_scaled = loaded_scaler.transform(
    X_test_known_processed[selected_feature_names]
)

known_if_scores = loaded_if.score_samples(known_if_scaled)

known_if = (
    known_if_scores < improved_if_threshold
).astype(int)

known_systems = {
    "Random Forest": rf_known,
    "XGBoost": xgb_known,
    "RF + XGBoost": ((rf_known == 1) | (xgb_known == 1)).astype(int),
    "Two-Stage Hybrid": (
        (xgb_known == 1) | (known_if == 1)
    ).astype(int)
}

rows = []

for name, pred in known_systems.items():

    tn, fp, fn, tp = confusion_matrix(
        y_known_test, pred, labels=[0, 1]
    ).ravel()

    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_known_test, pred),
        "Precision": precision_score(y_known_test, pred, zero_division=0),
        "Recall": recall_score(y_known_test, pred, zero_division=0),
        "F1": f1_score(y_known_test, pred, zero_division=0),
        "FPR": fp / (fp + tn) if (fp + tn) else 0
    })

known_test_results = pd.DataFrame(rows).sort_values(
    "F1", ascending=False
)

print("HELD-OUT KNOWN-TRAFFIC RESULTS")
display(known_test_results.round(4))


# Evaluate withheld PortScan category separately

portscan_rf = rf_model.predict(X_zero_day_processed)

portscan_xgb_prob = xgb_model.predict_proba(
    X_zero_day_processed
)[:, 1]

portscan_xgb = (
    portscan_xgb_prob >= xgb_threshold
).astype(int)

portscan_scaled = loaded_scaler.transform(
    X_zero_day_processed[selected_feature_names]
)

portscan_scores = loaded_if.score_samples(portscan_scaled)

portscan_if = (
    portscan_scores < improved_if_threshold
).astype(int)

portscan_systems = {
    "Random Forest": portscan_rf,
    "XGBoost": portscan_xgb,
    "RF + XGBoost": (
        (portscan_rf == 1) | (portscan_xgb == 1)
    ).astype(int),
    "Isolation Forest": portscan_if,
    "Two-Stage Hybrid": (
        (portscan_xgb == 1) | (portscan_if == 1)
    ).astype(int)
}

portscan_rows = []

for name, pred in portscan_systems.items():

    detected = int(np.sum(pred == 1))

    portscan_rows.append({
        "Model": name,
        "PortScan Detected": detected,
        "PortScan Missed": len(pred) - detected,
        "Detection Rate": detected / len(pred) if len(pred) else 0
    })

portscan_results = pd.DataFrame(portscan_rows).sort_values(
    "Detection Rate", ascending=False
)

print("\nWITHHELD PORTSCAN CATEGORY RESULTS")
display(portscan_results.round(4))

print("\nFINAL EVALUATION COMPLETED")
print("Known traffic and PortScan results are reported separately.")
print("No thresholds were adjusted using the held-out test data.")

In [ ]:
# ============================================================
# STEP 42: EXPORT FINAL PROJECT RESULTS
# ============================================================

import os
import pandas as pd
from sklearn.metrics import confusion_matrix

results_path = str(PROJECT_DIR / "output" / "final_results")
os.makedirs(results_path, exist_ok=True)

# 1. Validation results
comparison.to_csv(
    f"{results_path}/validation_model_comparison.csv",
    index=False
)

# 2. Held-out known traffic results
known_test_results.to_csv(
    f"{results_path}/heldout_known_traffic_results.csv",
    index=False
)

# 3. Withheld PortScan results
portscan_results.to_csv(
    f"{results_path}/withheld_portscan_results.csv",
    index=False
)

# 4. Validation confusion matrix
validation_cm = confusion_matrix(
    y_val,
    alarm_pred,
    labels=[0, 1]
)

pd.DataFrame(
    validation_cm,
    index=["Actual Benign", "Actual Attack"],
    columns=["Predicted Benign", "Predicted Attack"]
).to_csv(f"{results_path}/validation_confusion_matrix.csv")

# 5. Held-out test confusion matrices
for name, predictions in {
    "XGBoost": xgb_known,
    "Two_Stage_Hybrid": (
        (xgb_known == 1) | (known_if == 1)
    ).astype(int)
}.items():

    cm = confusion_matrix(
        y_known_test,
        predictions,
        labels=[0, 1]
    )

    pd.DataFrame(
        cm,
        index=["Actual Benign", "Actual Attack"],
        columns=["Predicted Benign", "Predicted Attack"]
    ).to_csv(
        f"{results_path}/{name}_heldout_confusion_matrix.csv"
    )

print("=" * 65)
print("FINAL PROJECT RESULTS EXPORTED SUCCESSFULLY")
print("=" * 65)

for file in sorted(os.listdir(results_path)):
    print(file)

print("\nSaved location:")
print(results_path)